

# Transforms

An :class:`~neuralset.events.transforms.EventsTransform` modifies the
events DataFrame — filtering rows, adding columns, splitting into
train/test, linking text context, etc.  Transforms sit between a
:term:`Study` (which produces events) and :term:`extractors <Extractor>`
(which consume them).


## Applying a Transform

Load events from a study, then apply a transform.
:class:`~neuralset.events.transforms.QueryEvents` is the most common
one — it filters events after the study has been fully loaded and
cached.  (Compare with ``Study(query=...)`` which filters *timelines*
before loading — coarser but avoids building unwanted timelines.)
Because it's fast, ``QueryEvents`` itself doesn't need caching:



In [ ]:
import neuralset as ns
from neuralset.events import transforms

study = ns.Study(name="Fake2025Meg", path=ns.CACHE_FOLDER)
events = study.run()
print(f"Before: {len(events)} events, types: {events.type.unique().tolist()}")

In [ ]:
query = transforms.QueryEvents(query="type in ['Word', 'Meg']")
events = query(events)

print(f"After QueryEvents: {len(events)} events, types: {events.type.unique().tolist()}")

## In Practice: Chain

Applying transforms one-by-one is fine for exploration, but in real
configs you want a single object that bundles a study with its
transforms.  :class:`~neuralset.base.Chain` sequences a study and
transforms into a reproducible, cacheable pipeline:



In [ ]:
chain = ns.Chain(
    steps=[
        {"name": "Fake2025Meg", "path": str(ns.CACHE_FOLDER)},
        {"name": "QueryEvents", "query": "type in ['Word', 'Meg']"},
    ],
)
events = chain.run()
print(f"Chain: {len(events)} events, types: {events.type.unique().tolist()}")

The dict form (``{"name": "QueryEvents", "query": "..."}`` ) is
equivalent to ``QueryEvents(query="...")`` — it dispatches by name.
This makes configs easy to serialize to YAML or JSON.

.. seealso::

   :doc:`06_chains` — full details on ``Chain`` composition, caching
   placement, and how to configure pipelines effectively via ``Step``
   fields.



## Available Transforms

.. list-table::
   :header-rows: 1
   :widths: 30 70

   * - Transform
     - Purpose
   * - :class:`~neuralset.events.transforms.QueryEvents`
     - Filter events using a pandas query string
   * - :class:`~neuralset.events.transforms.RemoveMissing`
     - Drop events with missing values in a given field
   * - :class:`~neuralset.events.transforms.CreateColumn`
     - Add a column with a default value, optionally update rows via query
   * - :class:`~neuralset.events.transforms.SelectIdx`
     - Select events by unique-value index in a column
   * - :class:`~neuralset.events.transforms.AlignEvents`
     - Cross-subject or cross-modality alignment
   * - :class:`~neuralset.events.transforms.EnsureTexts`
     - Guarantee ``Text`` events exist (reconstruct from Words if needed)
   * - :class:`~neuralset.events.transforms.AddSentenceToWords`
     - Match words to sentences, create ``Sentence`` events
   * - :class:`~neuralset.events.transforms.AddContextToWords`
     - Build a causal ``context`` field for each word
   * - :class:`~neuralset.events.transforms.AssignWordSplitAndContext`
     - Full pipeline: sentence assignment + split + context
   * - :class:`~neuralset.events.transforms.AssignSentenceSplit`
     - Train/val/test splitting by sentence (prevents data leakage)
   * - :class:`~neuralset.events.transforms.AssignKSplits`
     - K-fold splitting
   * - :class:`~neuralset.events.transforms.ChunkEvents`
     - Split long media events into fixed-duration chunks
   * - :class:`~neuralset.events.transforms.ConfigureEventLoader`
     - Modify loading parameters for events with dynamic filepaths
   * - :class:`~neuralset.events.transforms.ExtractAudioFromVideo`
     - Extract audio tracks from Video events as separate Audio events



## Caching

Transforms support caching via the ``infra`` parameter. This avoids
recomputing expensive transforms (e.g. text processing) on every run:

```python
transform = transforms.AddSentenceToWords(
    infra={"backend": "Cached", "folder": "/cache"},
)
events = transform(events)  # cached on subsequent calls
```
See [exca](https://github.com/facebookresearch/exca) for all
caching options.



## Advanced: Alignment and Chunking

**Alignment** —
Suppose three subjects each heard the word "cat" at different times.
You want to compare or **average their brain responses** to that word.
:class:`~neuralset.events.transforms.AlignEvents` does this by
regrouping events: for each unique stimulus (here each word), it
builds a virtual :term:`timeline <Timeline>` — as if all subjects had
been recorded in the same session — by gathering their neural
recordings together and time-shifting them so the stimulus starts
at ``t=0``.

After alignment you can segment normally and the resulting dataset
will have one segment per stimulus × subject, ready for cross-subject
averaging or comparison.

```python
transforms.AlignEvents(
    trigger_type="Word",
    trigger_field="text",
    types_to_align="Meg",
)
```
**Chunking** —
:class:`~neuralset.events.transforms.ChunkEvents` splits long media
events (``Audio``, ``Video``) into shorter segments, both to avoid
out-of-memory errors and to prevent data leakage when extractors
(e.g. transformer models) see more context than the segment window:

```python
transforms.ChunkEvents(
    event_type_to_chunk="Audio",
    max_duration=5.0,        # seconds
    min_duration=2.0,        # avoid tiny leftover chunks
)
```


## Create Your Own

Subclass :class:`~neuralset.events.transforms.EventsTransform` and
implement ``_run(events) -> events``:



In [ ]:
import pandas as pd


class DropShortEvents(transforms.EventsTransform):
    min_duration: float = 0.1

    def _run(self, events: pd.DataFrame) -> pd.DataFrame:
        return events.loc[events["duration"] >= self.min_duration]


demo_events = ns.events.standardize_events(
    pd.DataFrame(
        [
            dict(type="Word", start=1.0, duration=0.3, text="hello", timeline="run-01"),
            dict(type="Word", start=2.0, duration=0.05, text="x", timeline="run-01"),
            dict(type="Action", start=3.0, duration=0.02, timeline="run-01"),
        ]
    )
)

filtered = DropShortEvents(min_duration=0.1).run(demo_events)
print(f"{len(demo_events)} -> {len(filtered)} events")

## Next Steps

- Extract features from events: :doc:`04_extractors`
- Full pipeline composition with Chain: :doc:`06_chains`

